# Chapter 8 — Using GitHub Copilot with the QDK

**Learning objectives**
- Set up GitHub Copilot alongside the QDK VS Code extension
- Use Copilot completions effectively for Q# operations, functors, and stdlib patterns
- Use Copilot Chat to explain, debug, and refactor Q# code
- Write prompts that produce correct Q# (and know where Copilot needs guidance)
- Use Copilot to accelerate Python driver code for Q# workflows
- Understand Copilot's limitations with quantum-specific semantics

## Setup

This chapter is primarily a VS Code workflow, but the cells below provide runnable Q# examples matching the patterns covered. Read each section's VS Code guidance, then run the cells to verify the output.

In [ ]:
import qsharp
from qsharp_widgets import Circuit, Histogram
print(f"qsharp {__import__('importlib.metadata', fromlist=['version']).version('qsharp')}")

## 8.1 Setup: Copilot + QDK in VS Code

Both extensions work side-by-side in VS Code. Install order doesn't matter.

**Required extensions:**
1. **GitHub Copilot** (`GitHub.copilot`) — completions
2. **GitHub Copilot Chat** (`GitHub.copilot-chat`) — `/explain`, `/fix`, `/doc`, inline chat
3. **Azure Quantum Development Kit** (`quantum.qsharp-lang-vscode`) — Q# language server, syntax, debugging

**Recommended VS Code settings (`settings.json`):**
```json
{
    "github.copilot.enable": {
        "qsharp": true,
        "python": true
    },
    "editor.inlineSuggest.enabled": true,
    "github.copilot.inlineSuggest.enable": true
}
```

**Verify it's working:** Open a `.qs` file, type `operation` and wait ~1 second — Copilot should suggest a signature. If it doesn't, check that the QDK extension is active (bottom status bar shows `Q#`).

## 8.2 Effective Completions: What Copilot Gets Right

Copilot has seen substantial Q# code and handles these patterns well:

- **Boilerplate operations**: signatures, `use` allocation, `Reset`/`ResetAll`
- **Standard gates and measurements**: `H`, `CNOT`, `MResetZ`, `MeasureEachZ`
- **`ApplyToEach` and higher-order patterns**
- **Adjoint/Controlled functor declarations**: `is Adj + Ctl`
- **Python driver code**: `qsharp.run()`, `qsharp.eval()`, result parsing

**How to get good completions:**
- Write a descriptive comment before the operation — Copilot uses it heavily
- Include the operation name in the comment when possible
- Accept partial suggestions with `Tab`, then let it continue

The cells below show patterns where Copilot completions are reliable — study these to recognize good suggestions vs ones that need editing.

In [ ]:
%%qsharp

// Copilot handles these patterns well

// Prepare an n-qubit GHZ state
operation PrepareGHZ(qs : Qubit[]) : Unit is Adj + Ctl {
    H(qs[0]);
    for i in 1..Length(qs) - 1 {
        CNOT(qs[0], qs[i]);
    }
}

// Apply a phase kickback oracle: marks |target⟩ with a phase flip
operation PhaseOracle(target : Qubit[], markedItem : Int) : Unit is Adj {
    if markedItem >= 0 and markedItem < 2 ^ Length(target) {
        within {
            ApplyXorInPlace(markedItem, target);
        } apply {
            Controlled Z(target[0..Length(target)-2], target[Length(target)-1]);
        }
    }
}

// Measure all qubits and return results as a tuple (Bool[], Int)
operation MeasureAndDecode(qs : Qubit[]) : (Bool[], Int) {
    let results = MeasureEachZ(qs);
    ResetAll(qs);
    let bits = [r == One | r in results];
    mutable value = 0;
    for (i, b) in Microsoft.Quantum.Arrays.Enumerated(bits) {
        if b { set value = value + 2^i; }
    }
    (bits, value)
}

In [ ]:
# Copilot is also strong on Python driver code patterns like these:

def run_and_count(expr: str, shots: int = 200) -> dict:
    """Run a Q# expression and return a counts dict."""
    results = qsharp.run(expr, shots)
    from collections import Counter
    return dict(Counter(str(r) for r in results))

def most_likely(expr: str, shots: int = 500) -> str:
    """Return the most frequent measurement outcome."""
    counts = run_and_count(expr, shots)
    return max(counts, key=counts.get)

# Verify
print(run_and_count("PrepareGHZ(Qubit[3]) -> ..."[:1]))  # just show the helpers exist
print("Helper functions defined.")

## 8.3 Copilot Chat: /explain and /fix on Q# Code

**Copilot Chat** (the sidebar panel, or `Ctrl+Shift+I`) understands Q# with the QDK extension active. The most useful commands:

| Command | When to use |
|---------|-------------|
| `/explain` | Paste a Q# operation and ask what it does |
| `/fix` | Paste a compiler error + surrounding code |
| `/doc` | Generate XML doc comments for an operation |
| `@workspace` | Ask about the whole Q# project |
| Inline chat (`Ctrl+I`) | Fix/refactor selected code in-place |

**Practical workflow for debugging:**
1. Copy the Q# compiler error from the VS Code Problems panel
2. Open Copilot Chat, paste the error + the offending operation
3. Ask: *"What is causing this error and how do I fix it?"*

The cells below show common Q# errors — practice explaining them to Copilot Chat.

In [ ]:
# These are the kinds of errors to paste into Copilot Chat for /fix:
# (They won't compile — they're shown as strings for reference)

error_examples = [
    {
        "error": "Cannot apply Adjoint to operation without Adj functor",
        "bad_code": """
operation MyOp(q : Qubit) : Unit {  // Missing `is Adj`
    H(q);
}
// Later:
Adjoint MyOp(q);  // Error: MyOp doesn't support Adj
""",
        "fix": "Add `is Adj` or `is Adj + Ctl` to the operation signature"
    },
    {
        "error": "Type mismatch: expected Double, got Int",
        "bad_code": """
let angle = 3;          // Int
Rz(angle, q);           // Rz expects Double
""",
        "fix": "Use `IntAsDouble(3)` or a literal `3.0`"
    },
    {
        "error": "Cannot update variable: variables are immutable by default",
        "bad_code": """
let count = 0;
set count = count + 1;  // Error: `let` bindings are immutable
""",
        "fix": "Declare with `mutable count = 0;` instead of `let`"
    },
]

for ex in error_examples:
    print(f"Error: {ex['error']}")
    print(f"Fix:   {ex['fix']}\n")

## 8.4 Prompt Engineering for Q# Code Generation

Copilot generates better Q# when your prompts include:

1. **The qubit layout** — e.g., "takes `n` data qubits and 1 ancilla"
2. **The expected functor** — e.g., "must support Adjoint"
3. **The stdlib namespace** — e.g., "use `Std.Arrays`" or "use `Microsoft.Quantum.Arithmetic`"
4. **The return type** — Q# is strongly typed; stating it avoids type errors
5. **Concrete examples** — e.g., "for n=3 it should produce |000⟩ + |111⟩"

Compare these two prompts:

**Weak:** `// write a Grover oracle`  
**Strong:** `// Phase oracle for Grover search: marks |target⟩ by flipping its phase. Takes Qubit[] register and Int markedItem. Must support Adjoint.`

The cells below show operations generated from strong prompts — verify they compile and run correctly.

In [ ]:
%%qsharp

// Strong prompt result: Grover diffusion operator.
// Takes Qubit[] register, applies the inversion-about-average operator.
// Must support Adjoint (it's self-adjoint, so Adjoint = identity).
operation GroverDiffusion(register : Qubit[]) : Unit is Adj {
    ApplyToEachA(H, register);
    ApplyToEachA(X, register);
    Controlled Z(register[0..Length(register)-2], register[Length(register)-1]);
    ApplyToEachA(X, register);
    ApplyToEachA(H, register);
}

// Strong prompt result: one Grover iteration.
// Applies oracle then diffusion to Qubit[] register.
// markedItem: the index to search for.
operation GroverIteration(register : Qubit[], markedItem : Int) : Unit {
    // Phase oracle: mark target item
    within {
        ApplyXorInPlace(markedItem, register);
    } apply {
        Controlled Z(register[0..Length(register)-2], register[Length(register)-1]);
    }
    GroverDiffusion(register);
}

// Run Grover search for markedItem in an n-qubit register
operation GroverSearch(n : Int, markedItem : Int, iterations : Int) : Int {
    use qs = Qubit[n];
    ApplyToEach(H, qs);
    for _ in 1..iterations {
        GroverIteration(qs, markedItem, );
    }
    let result = MeasureInteger(qs);
    ResetAll(qs);
    result
}

In [ ]:
from collections import Counter

# Search for item 5 in 3-qubit (8-element) space, 1 Grover iteration
results = qsharp.run("GroverSearch(3, 5, 1)", shots=200)
counts = Counter(str(r) for r in results)
print("Grover search results (looking for 5):")
for outcome, count in sorted(counts.items(), key=lambda x: -x[1])[:5]:
    print(f"  {outcome}: {count}/200")
Histogram(results)

## 8.5 Where Copilot Needs Correction

Copilot makes predictable mistakes with Q#. Know these so you can spot and fix them quickly:

| Mistake | What Copilot does | Correct approach |
|---------|------------------|-----------------|
| **Lambda type annotations** | `(x : Int) -> x + 1` | `x -> x + 1` (no annotations in lambdas) |
| **Int/Double mixing** | `Rx(3, q)` | `Rx(3.0, q)` or `Rx(IntAsDouble(n), q)` |
| **`IntAsDouble` namespace** | `Math.IntAsDouble` | `Std.Convert.IntAsDouble` |
| **Mutable vs immutable** | `let x = 0; set x = 1;` | `mutable x = 0; set x = 1;` |
| **Missing `ResetAll`** | Forgets to reset qubits | Always reset before `use` block ends |
| **`qsharp.__version__`** | Tries to read version attribute | `importlib.metadata.version('qsharp')` |
| **`EstimatesOverview(list)`** | Passes a Python list | Pass a single result object |

The cell below demonstrates the lambda annotation mistake — a common Copilot error:

In [ ]:
%%qsharp

// Copilot sometimes generates: (x : Int) -> x * 2  -- this FAILS
// Correct Q# lambda syntax has no type annotations:

function DoubleAll(arr : Int[]) : Int[] {
    // Correct: no type annotation on lambda parameter
    Microsoft.Quantum.Arrays.Mapped(x -> x * 2, arr)
}

DoubleAll([1, 2, 3, 4, 5])

In [ ]:
%%qsharp

// Copilot may suggest Math.IntAsDouble — wrong namespace.
// Correct: Std.Convert.IntAsDouble

import Std.Convert.IntAsDouble;

function AngleForStep(step : Int, total : Int) : Double {
    2.0 * Microsoft.Quantum.Math.PI() * IntAsDouble(step) / IntAsDouble(total)
}

AngleForStep(1, 8)  // Should be π/4 ≈ 0.785

## 8.6 Accelerating Python Interop with Copilot

Copilot is most reliable for the Python side of QDK workflows — parsing results, plotting, parameter sweeps. The pattern that works well:

1. Write the Q# operation manually (you know the quantum semantics)
2. Let Copilot write the Python driver (boilerplate it knows well)

Good Copilot prompts for Python drivers:
- `# Run BellState 500 times with depolarizing noise p, return fidelity`
- `# Sweep noise from 0 to 0.2 and plot fidelity vs noise`
- `# Parse qsharp.run results and return counts dict`

In [ ]:
%%qsharp

// Q# side: written by hand
operation RandomWalk(steps : Int) : Int {
    use qs = Qubit[1];
    mutable position = 0;
    for _ in 1..steps {
        H(qs[0]);
        let r = MResetZ(qs[0]);
        set position = r == One ? position + 1 | position - 1;
    }
    position
}

In [ ]:
# Python driver: Copilot generates this well from a comment like:
# "Run RandomWalk for steps in [10,20,50,100], plot distribution of final positions"

import matplotlib.pyplot as plt
import numpy as np

step_counts = [10, 20, 50, 100]
fig, axes = plt.subplots(1, len(step_counts), figsize=(14, 3), sharey=False)

for ax, steps in zip(axes, step_counts):
    results = qsharp.run(f"RandomWalk({steps})", shots=300)
    positions = [r for r in results]
    ax.hist(positions, bins=range(min(positions)-1, max(positions)+2), density=True,
            color='steelblue', alpha=0.8, edgecolor='white')
    ax.set_title(f"{steps} steps")
    ax.set_xlabel("Final position")
    # Overlay expected Gaussian: std = sqrt(steps)
    x = np.linspace(min(positions)-1, max(positions)+1, 100)
    ax.plot(x, 1/np.sqrt(2*np.pi*steps) * np.exp(-x**2/(2*steps)),
            'r-', linewidth=2, label='Theory')

axes[0].set_ylabel("Probability density")
fig.suptitle("Quantum Random Walk: Position Distribution")
plt.tight_layout()
plt.show()

## 8.7 Using Copilot Chat for Code Review and Documentation

Beyond generation, Copilot Chat is useful for reviewing Q# code you've written:

**Useful Chat prompts for Q# code review:**
- *"Does this operation correctly implement a phase kickback oracle? What edge cases might fail?"*
- *"This QFT implementation looks wrong — can you identify the bug?"*
- *"Generate XML documentation comments for this operation"*
- *"Is there a stdlib function in `Std.Arrays` or `Std.Math` that does what this function does?"*
- *"Rewrite this loop using `ApplyToEach` or a higher-order function"*

**For migration from Qiskit:**
- *"I have this Qiskit circuit — translate it to Q# operations"*
- *"What is the Q# equivalent of `QuantumCircuit.measure_all()`?"*
- *"How does Q# qubit allocation differ from Qiskit? Explain the `use` statement"*

In [ ]:
%%qsharp

// Example: Ask Copilot Chat to review this SWAP test implementation
// Prompt: "Is this a correct SWAP test? What does it measure?"

operation SwapTest(qs1 : Qubit[], qs2 : Qubit[]) : Result {
    use ancilla = Qubit();
    H(ancilla);
    for i in 0..Length(qs1)-1 {
        Controlled SWAP([ancilla], (qs1[i], qs2[i]));
    }
    H(ancilla);
    MResetZ(ancilla)
}

// Run SWAP test between identical states (should give Zero with high prob)
operation SwapTestIdentical() : Result {
    use q1 = Qubit[2];
    use q2 = Qubit[2];
    // Prepare identical states
    H(q1[0]); CNOT(q1[0], q1[1]);
    H(q2[0]); CNOT(q2[0], q2[1]);
    let result = SwapTest(q1, q2);
    ResetAll(q1 + q2);
    result
}

// Run SWAP test between orthogonal states (should give Zero with prob 0.5)
operation SwapTestOrthogonal() : Result {
    use q1 = Qubit[1];
    use q2 = Qubit[1];
    // |0⟩ and |1⟩ — orthogonal
    X(q2[0]);
    let result = SwapTest(q1, q2);
    ResetAll(q1 + q2);
    result
}

In [ ]:
from collections import Counter

identical = qsharp.run("SwapTestIdentical()", shots=300)
orthogonal = qsharp.run("SwapTestOrthogonal()", shots=300)

print("SWAP test on identical states (expect ~100% Zero):")
print(Counter(str(r) for r in identical))

print("\nSWAP test on orthogonal states (expect ~50% Zero):")
print(Counter(str(r) for r in orthogonal))

## 8.8 Copilot for Multi-File Q# Projects

When working with VS Code Q# projects (`.csproj` + multiple `.qs` files), Copilot Chat with `@workspace` can reason across the whole project:

- `@workspace What operations are defined in the Chemistry namespace?`
- `@workspace Where is the Jordan-Wigner encoding implemented?`
- `@workspace Find all operations that don't have XML documentation comments`

**Project structure Copilot understands well:**
```
MyProject/
├── MyProject.csproj
├── src/
│   ├── Oracles.qs       # namespace MyProject.Oracles
│   ├── Algorithms.qs    # namespace MyProject.Algorithms
│   └── Utils.qs         # namespace MyProject.Utils
└── tests/
    └── Tests.qs         # namespace MyProject.Tests
```

**Tip:** Add a `// Namespace: MyProject.Oracles — phase kickback oracles for Grover search` comment at the top of each `.qs` file. Copilot uses these as context when generating code that imports or calls across files.

**Generating a `.csproj` file:** Ask Copilot Chat:
> *"Generate a Q# project file for a library project named MyAlgorithms targeting QDK 1.x"*

## 8.9 Copilot Limitations to Keep in Mind

Q# is a low-volume language compared to Python or TypeScript — Copilot's training data is sparser, and it occasionally generates plausible-looking but incorrect code. The most common failure modes:

**Semantically wrong but syntactically valid:**
- Applying gates in the wrong order (e.g., CNOT before H in Bell prep)
- Forgetting to reset qubits — silent correctness bug in multi-shot runs
- Using `let` when `mutable` is needed, or vice versa
- Wrong functor variant: using `Adjoint` of a non-self-adjoint gate

**Rule of thumb:** Always run Copilot-generated Q# operations through `DumpMachine()` or a histogram check before trusting the output. For any operation involving adjoints or controlled variants, verify with the circuit diagram (§3.1–3.4).

**What Copilot cannot do:**
- Verify quantum correctness (it doesn't simulate)
- Know your hardware's native gate set or connectivity
- Predict `qsharp.run()` output — always run it
- Replace reading the [Q# language reference](https://learn.microsoft.com/azure/quantum/user-guide/) for subtle semantics

## Exercise: Copilot-Assisted Implementation

Use Copilot to implement the following, then verify correctness:

1. **In VS Code:** Write the comment `// Prepare a uniform superposition over the first k basis states out of 2^n, where k <= 2^n` and let Copilot complete the operation. Verify it with `DumpMachine()`.

2. **In VS Code:** Ask Copilot Chat: *"I have a Q# operation BellState() that returns Result[]. Write Python code to run it 1000 times with BitFlipNoise at p=0.01, 0.05, 0.1 and plot the error rate for each."* Run the generated code and check it matches Chapter 3's fidelity curve.

3. **In VS Code:** Find a Q# operation from Chapter 5 or 6 and paste it into Copilot Chat with the prompt *"Add XML documentation comments to this operation."* Review the generated docs for accuracy.

In [ ]:
# Verification cell for exercise 1: DumpMachine check
# Replace the body with whatever Copilot generated

%%qsharp

import Std.Diagnostics.DumpMachine;

// YOUR COPILOT-GENERATED OPERATION HERE
// Example reference implementation to compare against:
operation UniformKBasisStates(n : Int, k : Int) : Unit {
    use qs = Qubit[n];
    // Amplitude for each of the k states: 1/sqrt(k)
    // Simple approach: prepare |0..0⟩ and rotate
    // (A full correct implementation uses QRAM or recursive amplitude estimation)
    // For k = 2^n, this is just ApplyToEach(H, qs)
    if k == 2^n {
        ApplyToEach(H, qs);
        Message($"Uniform superposition over all {2^n} states:");
        DumpMachine();
    } else {
        Message($"Non-power-of-2 k={k} requires custom state prep — see qsharp.estimate patterns");
    }
    ResetAll(qs);
}

In [ ]:
qsharp.eval("UniformKBasisStates(3, 8)")

## Summary

- Install **GitHub Copilot** + **GitHub Copilot Chat** alongside the **Azure QDK** extension; enable Q# in `settings.json`
- Copilot is reliable for boilerplate Q# (gates, measurements, `ApplyToEach`, functor declarations) and Python drivers
- Use **descriptive comments** before operations — include qubit layout, functor requirements, and return type
- Use **Copilot Chat** `/explain` and `/fix` for debugging compiler errors; `@workspace` for project-level questions
- Watch for predictable mistakes: lambda type annotations, `Int`/`Double` mixing, wrong `IntAsDouble` namespace, missing `ResetAll`
- Best workflow: **write Q# by hand** (you own the quantum semantics), **let Copilot write Python** (it knows the driver patterns)
- Always verify generated Q# with `DumpMachine()` or a histogram — Copilot cannot simulate quantum correctness